# Exercise: Delta threshold

The stop decision uses a delta threshold (default 0.01) to determine if graph features add value.

**Your task:**
1. Run the comparison with default threshold (0.01)
2. Change the threshold to 0.005 and 0.02
3. Observe how the stop decision changes

What does this tell you about the sensitivity of the decision?

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent / "src"))

import numpy as np
from phase01_lightgbm_baseline.data_loader import DataLoader
from phase01_lightgbm_baseline.lightgbm_baseline import LightGBMWrapper
from phase01_lightgbm_baseline.graph_features import GraphFeatureExtractor
from phase01_lightgbm_baseline.comparison import compute_comparison_report, check_significance, generate_stop_decision

# Load data
loader = DataLoader()
train_df, test_df = loader.load_data(sample_limit=500)

# Train baseline model
baseline_features = ["TransactionAmt", "time_since_creation", "n_prior_transactions"]
X_train_base = train_df[baseline_features].values
y_train = train_df["isFraud"].values
X_test_base = test_df[baseline_features].values
y_test = test_df["isFraud"].values

baseline_model = LightGBMWrapper(random_state=42)
baseline_metrics = baseline_model.full_pipeline(X_train_base, y_train, X_test_base, y_train, X_test_base, y_test)

# Train graph model
extractor = GraphFeatureExtractor()
train_graph, test_graph = extractor.append_graph_features(train_df.copy(), test_df.copy())
graph_feature_cols = ["user_degree", "merchant_degree", "user_clustering"]
all_features = baseline_features + graph_feature_cols
X_train_graph = train_graph[all_features].values
X_test_graph = test_graph[all_features].values

graph_model = LightGBMWrapper(random_state=42)
graph_metrics = graph_model.full_pipeline(X_train_graph, y_train, X_test_graph, y_train, X_test_graph, y_test)

# Compute deltas
deltas = compute_comparison_report(baseline_metrics, graph_metrics)
delta_pr_auc = deltas["delta_pr_auc"]

# Test significance
raw_probs_baseline = baseline_model.predict(X_test_base)
raw_probs_graph = graph_model.predict(X_test_graph)
p_value = check_significance(raw_probs_baseline, raw_probs_graph)

## Your solution

Test different thresholds and observe how the stop decision changes.

In [ ]:
# Your code here



## Solution

```python
print(f"Delta PR-AUC: {delta_pr_auc:+.4f}")
print(f"P-value: {p_value:.4f}")
print()

thresholds = [0.005, 0.01, 0.02]
for threshold in thresholds:
    decision = generate_stop_decision(delta_pr_auc, p_value, delta_threshold=threshold)
    print(f"Threshold {threshold}: {decision}")
```

**Observation:** Lower thresholds make it easier to continue (graphs add value), while higher thresholds require larger improvements. The choice depends on the cost of graph feature computation vs. the expected benefit.